# Homework №3


## Effective Flow Matching Training (10 pt)


In this homework, you are asked to:

1) Implement a pixel-space flow matching model ([JiT](https://arxiv.org/abs/2511.13720)) **(6 pt)**

2) Adapt it to the latent diffusion model (i.e., run it in VAE space) **(1 pt)**

3) Implement [REPA](https://openreview.net/forum?id=DJSZGGZYVi) to enforce diffusion to learn internal features faster **(3 pt)**



### Install if required

In [ ]:
! gdown https://drive.google.com/uc?id=1EepHAfl9i9ThhRmpmlCNb5Oht3YLrtWu
! gdown https://drive.google.com/uc?id=1XFHYcPZJTO7VcVOhAH30wJHose6Tb9rM
! gdown https://drive.google.com/uc?id=1NyoBpbjAsBwdAsPdBU0jZvUUD1-q7gb-
! gdown https://drive.google.com/uc?id=1i-tGtvY57E4s1BcrwxAznTZ5HIHCHEIV
! tar -xf stanford_cars.tar
! tar -xf stanford_cars_flat.tar

In [ ]:
! pip install torch_fidelity

### Import libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import math
import sys
import torch_fidelity
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from transformers import AutoModel

## Flow matching implementation (6 pt)

Implement flow matching training and sampling for the pixel-space diffusion transformer, [JiT](https://arxiv.org/abs/2511.13720). 

**Forward process:** $(1-t) x + t \epsilon$

**v-loss:** $\lVert v_\theta - v \rVert^2_2$  

In [ ]:
from model_jit import JiT_models


class Denoiser(nn.Module):
    def __init__(self, model='JiT-S/16', img_size=256, in_channels=3, class_num=196, label_drop_prob=0.1, P_mean=0.8, P_std=0.8, t_eps=5e-2):
        super().__init__()
        self.net = JiT_models[model](
            input_size=img_size,
            in_channels=in_channels,
            num_classes=class_num,
        )
        self.in_channels = in_channels
        self.img_size = img_size
        self.num_classes = class_num

        self.label_drop_prob = label_drop_prob
        self.P_mean = P_mean
        self.P_std = P_std
        self.t_eps = t_eps

    def drop_labels(self, labels):
        """
        Implement label dropping during training for CFG (use self.label_drop_prob)
        """
        # <YOUR CODE>
        pass

    def sample_t(self, n: int, device=None):
        """ 
        Implement timestep sampling distribution. 
        
        Use shifted logit normal with parameters self.P_std + self.P_mean
        """
        # <YOUR CODE>
        pass

    def forward(self, x, labels, pixel_image=None, dino_encoder=None, repa=None, extractor=None):
        """
        Implement flow matching loss calculation.
        1) Drop labels for CFG
        2) Apply forward process to x
        3) Select the proper prediction type for the model
        4) Compute v-loss
        5) Add the REPA loss (Advice: implement it when you approach the corresponding cell)
        
        Args:
            x: clean images (or latents)
            labels: class labels
            ----------
            REPA args
            ----------
            pixel_image: use it as a DINO input for REPA. Do not forget preprocess is using dino_preprocess_raw_image()
            dino_encoder: DINO encoder for REPA
            repa: REPA class object that implements the feature projector and REPA loss 
            extractor: Extracts intermediate diffusion activations. You can implement this another way - up to you.
        """
        # <YOUR CODE>

        # REPA loss
        if encoder is not None:
            # <YOUR CODE>
            pass

        return loss

    
    
@torch.no_grad()
def generate(model, labels, steps=25, cfg_scale=4.0, vae=None):
    """
    Implement a flow matching generation process. You are free to use any solver. Euler is okay.
    You can also use any timestep scheduler from the previous HW.
    
    Hints:
        1) Do not forget about CFG. We already set the okay default cfg_scale but you free to play with it
        2) Consider torch.autocast to float16 for faster sampling
        3) (For the latent model) Do not forget to denormalize latents before decoding
    """
    # <YOUR CODE>

    if vae is not None:
        # <YOUR CODE>
    return images

In [ ]:
# Recommended hyperparameters

device = torch.device('cuda')
lr = 5e-4
batch_size = 32

### Create dataset and dataloader

We train our model on the [Stanford Cars](https://huggingface.co/datasets/tanganke/stanford_cars) dataset. It contains 16183 images of 196 classes.


**Dataset class returns a clean image, precomputed VAE latent and label.**

Note that you do not need to encode images during latent diffusion training with the VAE encoder.

In [ ]:
##################
# NO CODING CELL #
##################


class ImageWithLatentDataset(Dataset):
    def __init__(self, image_folder, latents_file, transform=None):
        self.image_dataset = datasets.ImageFolder(image_folder, transform=transform)
        data = torch.load(latents_file)
        self.latents = data['latents']
        self.labels = data['labels']

    def __len__(self):
        return len(self.image_dataset)

    def __getitem__(self, idx):
        image, label = self.image_dataset[idx]
        latent = self.latents[idx]
        return image, latent, label

    
transform=transforms.Compose([
    transforms.PILToTensor(),
    transforms.Lambda(lambda x: x.float().div(127.5).sub(1.0))
])

dataset_train = ImageWithLatentDataset(
    image_folder='stanford_cars',
    latents_file='latents_stanford_cars.pt',
    transform=transform
)

print(f"Dataset size : {len(dataset_train)}")

data_loader_train = DataLoader(
    dataset_train,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
    shuffle=True,
)

### Evaluation function

You can use it as is. No coding here

In [ ]:
@torch.no_grad()
def evaluate(model, cfg_scale=3.0, batch_size=49, num_images=196, save_folder='samples', vae=None):
    assert num_images % batch_size == 0
    num_steps = num_images // batch_size

    print("Saving to:", save_folder)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    class_num = model.num_classes
    assert num_images % class_num == 0, "Number of images per class must be the same"
    class_label = np.arange(0, class_num).repeat(num_images // class_num)

    for i in range(num_steps):
        start_idx = batch_size * i
        end_idx = start_idx + batch_size
        labels_gen = class_label[start_idx:end_idx]
        labels_gen = torch.Tensor(labels_gen).long().cuda()

        sampled_images = generate(model, labels_gen, vae=vae, cfg_scale=cfg_scale)

        # denormalize images
        sampled_images = (sampled_images + 1) / 2
        sampled_images = sampled_images.clamp(0, 1)
        sampled_images = sampled_images.detach().cpu()

        # distributed save images
        for b_id in range(sampled_images.size(0)):
            img_id = i * sampled_images.size(0) + b_id

            arr = (sampled_images[b_id].permute(1, 2, 0).float().numpy() * 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(save_folder, f'{img_id:06d}.jpg'), optimize=False, compress_level=1)

    print('saved')

    # # compute FID and IS
    metrics_dict = torch_fidelity.calculate_metrics(
        input1=save_folder,
        input2='stanford_cars_flat',
        kid=True,
        verbose=False,
        kid_subset_size=196,
        kid_subsets=100,
        num_workers=2
    )

    kid_mean = metrics_dict['kernel_inception_distance_mean']
    return kid_mean

### Training function

It is already implemented **BUT**


It is **hightly recommened** (especially for colab/kaggle) to add **mixed-precision** for faster training. 

See the doc: https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html

Note that T4 does not support torch.bfloat16, use torch.float16

In [ ]:
def train_model(model, optimizer, max_iters, vae=None, encoder=None, repa=None, extractor=None):
    """
    Trains your flow matching model. 
    
    Consider adding mixed-precision. It will save your time. No other changes are needed.
    """
    
    model.train()

    losses = []
    KIDs = []

    loss_sum = 0
    loss_cnt = 0
    iteration = 0

    while True:
        for (x, latents, labels) in data_loader_train:
            iteration += 1
            latents = latents.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            x = x.to(device, non_blocking=True)

            # <YOUR CODE> Add mixed-precision training
            if encoder is not None:
                loss = model(latents, labels, x, encoder, repa, extractor)
            elif vae is not None:
                loss = model(latents, labels)
            else:
                loss = model(x, labels)

            loss_sum += loss.item()
            loss_cnt += 1

            # <YOUR CODE> Add mixed-precision
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            if iteration % 1000 == 0:
                torch.cuda.empty_cache()
                with torch.no_grad():
                    model.eval()
                    KID = evaluate(model, vae=vae)
                    KIDs.append(KID)
                    model.train()
                torch.cuda.empty_cache()

            if iteration % 100 == 0:
                model.eval()
                with torch.no_grad():
                    labels_gen = torch.Tensor([0,1,2,3,4,5,6,7]).long().cuda()

                    sampled_images = generate(model, labels_gen, vae=vae)

                    sampled_images = (sampled_images + 1) / 2
                    sampled_images = sampled_images.clamp(0, 1)
                    sampled_images = sampled_images.detach().cpu()

                model.train()

                losses.append(loss_sum/loss_cnt)
                loss_sum = 0
                loss_cnt = 0
                clear_output(wait=True)

                fig = plt.figure(figsize=(14, 6), constrained_layout=True)
                gs = gridspec.GridSpec(2, 8, figure=fig, height_ratios=[2, 1])

                ax1 = fig.add_subplot(gs[0, :4])
                ax1.semilogy(100 * torch.arange(1, len(losses)+1), losses)
                ax1.set_title('Loss')
                ax1.set_xlabel('Iteration')
                ax1.set_ylabel('Loss')
                ax1.grid()

                ax2 = fig.add_subplot(gs[0, 4:])
                ax2.plot(1000 * torch.arange(1, len(KIDs)+1), KIDs)
                ax2.set_title('KID')
                ax2.set_xlabel('Iteration')
                ax2.set_ylabel('KID mean')
                ax2.grid()

                for idx in range(8):
                    ax = fig.add_subplot(gs[1, idx])
                    ax.imshow((sampled_images[idx].permute(1, 2, 0).float().numpy() * 255).astype(np.uint8))
                    ax.axis('off')
                    ax.set_title(f'class {idx}', fontsize=8)

                plt.show()


            if iteration >= max_iters:
                break

        if iteration >= max_iters:
            break

### Using pixel space

In [ ]:
model = Denoiser()
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95))

In [ ]:
train_model(model, optimizer, 5000) # Consider less iteration for debuggind

## Latent diffusion model (1 pt)

1. Update the generation function to decode latents to images. Remember about denormalization.

2. You need to understand which parameters the Denoiser should take in this case. We want the model to operate on the same number of tokens as in the pixel space (256).

**Hint:** you can define a new patch size via the model name 'JiT-S/X', where X is a new patch size

In [ ]:
# Load the VAE model
from diffusers.models import AutoencoderKL
vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
None

In [ ]:
model = # <YOUR CODE>
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95))

In [ ]:
train_model(model, optimizer, 5000, vae=vae)

## REPA loss with [DINOv3](https://arxiv.org/abs/2508.10104) (3 pt)

Implement the **[improved REPA](https://arxiv.org/abs/2512.10794)**. You will need to implement the projection layer and the REPA loss.

In [ ]:
##################
# NO CODING CELL #
##################

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

def dino_preprocess_raw_image(x):
    x = (x + 1) / 2
    x = transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)(x)
    x = torch.nn.functional.interpolate(x, 256, mode='bicubic')
    return x

In [ ]:
# Load DINOv3: ViT-S/16

encoder = AutoModel.from_pretrained(
    "timm/vit_small_patch16_dinov3.lvd1689m"
    device_map="auto",
).eval()

### Extract diffusion features

Implement **ExtractorDiffusionFeatures** that extracts intermediate diffusion features.

Use hooks: https://docs.pytorch.org/docs/stable/generated/torch.Tensor.register_hook.html


**Hints:** 

1) The token sequence contains in_context and spatial tokens. You need only spatial ones
2) We recommend extracting features after the 6th block.

In [ ]:
class DiffusionFeatureExtractor:
    # <YOUR CODE>
    
    

class REPA(nn.Module):
    def __init__(self, net_hidden_size=384, **kwargs):
        """ 
        Create the projection layer according to the iREPA. 
        """
        
        super().__init__()
        
        # <YOUR CODE>
        
        
    def forward(self, dm_features, dino_features):
        """
        Implement the REPA loss using the projection model
        
        Args:
            dm_features [B, 256, D]: Extracted diffusion features
            dino_features [B, 256, D]: DINO features    
        """
        
        # <YOUR CODE>

        return repa_loss

In [ ]:
# Create the latent denoiser, extractor and repa. 

model = # <YOUR CODE>
model.to(device)
extractor =  DiffusionFeatureExtractor(model.net)
repa = <YOUR CODE>

optimizer = <YOUR CODE>

In [ ]:
# Consider less iteration for debugging

train_model(model, optimizer, 5000, vae=vae, encoder=encoder, repa=repa, extractor=extractor)

## Conclusions

Please discuss the design choices and techniques that led to noticeable improvements, such as faster convergence or better overall results.

**YOUR CONCLUSIONS**